# Pipe and Filter — Exemplo Executável

**Autores:** Thiago Leal, Emerson Silva, José Victor, Randson Bredley, Thiago Patriota, Gustavo Henrique (Grupo 3)

Este notebook demonstra o estilo arquitetural **Pipe and Filter** através de um pipeline de processamento de texto.
Mostramos a implementação correta (processamento em lote e streaming sob demanda com geradores), violações de restrições arquiteturais (acoplamento eferente e mutabilidade de payload), suas consequências mensuráveis e uma avaliação empírica de desempenho com `timeit`.

**Referências:**
- Shaw & Garlan, *Software Architecture: Perspectives on an Emerging Discipline*, 1996, cap. 2
- Bass, Clements & Kazman, *Software Architecture in Practice*, 4ª ed., 2021, cap. 13
- Buschmann et al., *Pattern-Oriented Software Architecture*, Vol. 1, 1996, cap. 2

## 1. Dependências

Este notebook usa apenas recursos da biblioteca padrão do Python — dispensando dependências externas.

In [1]:
# Dependências — apenas biblioteca padrão
import re
import copy
import timeit
from collections import Counter
from collections.abc import Iterator
from typing import Callable, Any

print("✅ Dependências carregadas com sucesso.")

✅ Dependências carregadas com sucesso.


## 2. Dados de entrada

Um texto bruto representativo sobre arquitetura de software que será processado ao longo do pipeline.

In [2]:
texto_bruto = """
A arquitetura de software define a estrutura fundamental de um sistema.
Ela determina como os componentes se organizam, como se comunicam e quais
restrições governam o projeto. Um estilo arquitetural como o Pipe and Filter
decompõe o processamento em etapas independentes, cada uma responsável por
uma transformação específica. Os filtros não conhecem uns aos outros;
eles apenas recebem dados, transformam e produzem resultados. Essa
independência permite que filtros sejam adicionados, removidos ou
substituídos sem afetar o restante do pipeline. O estilo é amplamente
utilizado em compiladores, pipelines de ETL, processamento de sinais
e sistemas de análise de logs. A simplicidade do modelo facilita a
compreensão e a manutenção, mas impõe custos de desempenho quando o
volume de dados é grande e a serialização entre filtros se torna
um gargalo.
"""

print(f"Texto de entrada ({len(texto_bruto.split())} palavras):")
print(texto_bruto.strip()[:200] + "...")

Texto de entrada (129 palavras):
A arquitetura de software define a estrutura fundamental de um sistema.
Ela determina como os componentes se organizam, como se comunicam e quais
restrições governam o projeto. Um estilo arquitetural ...


## 3. Implementação CORRETA — Pipe and Filter

Cada filtro é uma **função pura**: recebe dados de entrada, realiza uma transformação coerente e retorna um novo dado.
Nenhum filtro conhece os demais componentes da cadeia. A comunicação acontece exclusivamente através dos **pipes** (fluxo de dados conectado entre saída e entrada).

### 3.1 Definição dos filtros (Processamento em Lote)

In [3]:
# ---------- FILTRO 1: Normalizar ----------
def normalizar(texto: str) -> str:
    """Converte para minúsculas e remove pontuação."""
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', '', texto)
    return texto


# ---------- FILTRO 2: Tokenizar ----------
def tokenizar(texto: str) -> list[str]:
    """Divide o texto em palavras individuais."""
    return texto.split()


# ---------- FILTRO 3: Filtrar stop words ----------
STOP_WORDS_PT = {
    'a', 'o', 'e', 'é', 'de', 'do', 'da', 'dos', 'das', 'em', 'um',
    'uma', 'os', 'as', 'no', 'na', 'nos', 'nas', 'se', 'que', 'como',
    'por', 'para', 'com', 'não', 'nao', 'mais', 'ao', 'aos', 'sua',
    'seu', 'ela', 'ele', 'eles', 'elas', 'ou', 'uns', 'umas', 'entre',
    'cada', 'quando', 'esse', 'essa', 'este', 'esta', 'isso', 'isto',
}

def filtrar_stop_words(palavras: list[str]) -> list[str]:
    """Remove palavras comuns sem valor semântico."""
    return [p for p in palavras if p not in STOP_WORDS_PT and len(p) > 1]


# ---------- FILTRO 4: Contar frequências ----------
def contar_frequencias(palavras: list[str]) -> dict[str, int]:
    """Calcula a frequência de cada palavra (estágio sink / agregação)."""
    return dict(Counter(palavras).most_common())


print("✅ Quatro filtros definidos, cada um independente dos demais.")

✅ Quatro filtros definidos, cada um independente dos demais.


### 3.2 O pipeline (pipe)

O pipe conecta os filtros: a saída de um é canalizada para a entrada do próximo.
A função `pipeline` é genérica e compõe qualquer sequência de filtros compatíveis.

In [4]:
def pipeline(dado: Any, *filtros: Callable) -> Any:
    """Executa uma sequência de filtros, passando o resultado de um para o próximo."""
    resultado = dado
    for i, filtro in enumerate(filtros):
        resultado = filtro(resultado)
        nome = filtro.__name__ if hasattr(filtro, '__name__') else filtro.__class__.__name__
        print(f"  Filtro {i+1} ({nome}): {type(resultado).__name__} → {str(resultado)[:70]}...")
    return resultado


print("Executando pipeline CORRETO:")
print("=" * 60)
resultado_correto = pipeline(
    texto_bruto,
    normalizar,
    tokenizar,
    filtrar_stop_words,
    contar_frequencias
)
print("=" * 60)
print(f"\n📊 Top 10 palavras mais frequentes:")
for palavra, freq in list(resultado_correto.items())[:10]:
    print(f"   {palavra:.<25} {freq}")

Executando pipeline CORRETO:
  Filtro 1 (normalizar): str → 
a arquitetura de software define a estrutura fundamental de um sistem...
  Filtro 2 (tokenizar): list → ['a', 'arquitetura', 'de', 'software', 'define', 'a', 'estrutura', 'fu...
  Filtro 3 (filtrar_stop_words): list → ['arquitetura', 'software', 'define', 'estrutura', 'fundamental', 'sis...
  Filtro 4 (contar_frequencias): dict → {'filtros': 3, 'estilo': 2, 'processamento': 2, 'dados': 2, 'arquitetu...

📊 Top 10 palavras mais frequentes:
   filtros.................. 3
   estilo................... 2
   processamento............ 2
   dados.................... 2
   arquitetura.............. 1
   software................. 1
   define................... 1
   estrutura................ 1
   fundamental.............. 1
   sistema.................. 1


### 3.3 Reuso: montando outro pipeline com os mesmos filtros

A grande força do padrão Pipe and Filter: os mesmos filtros podem ser recombinados livremente para atender requisitos diferentes sem modificação de código.

In [5]:
# Pipeline alternativo: apenas normalizar e tokenizar (sem stop words, sem contagem)
print("Pipeline alternativo — apenas normalizar e tokenizar:")
print("=" * 60)
tokens = pipeline(texto_bruto, normalizar, tokenizar)
print(f"\nTotal de tokens: {len(tokens)}")
print(f"Primeiros 15: {tokens[:15]}")

Pipeline alternativo — apenas normalizar e tokenizar:
  Filtro 1 (normalizar): str → 
a arquitetura de software define a estrutura fundamental de um sistem...
  Filtro 2 (tokenizar): list → ['a', 'arquitetura', 'de', 'software', 'define', 'a', 'estrutura', 'fu...

Total de tokens: 129
Primeiros 15: ['a', 'arquitetura', 'de', 'software', 'define', 'a', 'estrutura', 'fundamental', 'de', 'um', 'sistema', 'ela', 'determina', 'como', 'os']


### 3.4 O Mecanismo do Pipe: Processamento sob Demanda com Geradores (`yield`)

No pipeline implementado anteriormente, cada filtro opera sobre a totalidade dos dados na memória de uma só vez (processamento em lote / *batch*), passando listas completas para a etapa seguinte. Para textos pequenos, isso é simples e eficiente.

No entanto, em sistemas reais de **análise contínua de logs e processamento massivo de dados (Big Data / ETL)**, carregar milhões de strings simultaneamente em listas no Python consome imensa quantidade de memória RAM, pois cada string aloca um objeto individual e a lista aloca um vetor de referências contíguo.

Em Python, a expressão mais fiel e performática do estilo **Pipe and Filter** para grandes volumes é obtida através de **geradores (`yield`)**. Com geradores, cada filtro consome a saída do filtro anterior sob demanda (**streaming item por item**), mantendo a pegada de memória RAM praticamente **constante (O(1))**, permitindo processar fluxos de dezenas de gigabytes sem sobrecarregar a memória.

In [6]:
# ---------- VERSÃO EM STREAMING COM GERADORES (yield) ----------

def normalizar_stream(linhas: Iterator[str]) -> Iterator[str]:
    """Emite linhas normalizadas uma a uma sob demanda."""
    for linha in linhas:
        linha_limpa = re.sub(r'[^\w\s]', '', linha.lower())
        if linha_limpa.strip():
            yield linha_limpa


def tokenizar_stream(linhas_stream: Iterator[str]) -> Iterator[str]:
    """Emite tokens sob demanda através de re.finditer, sem carregar listas inteiras na memória."""
    for linha in linhas_stream:
        for match in re.finditer(r'\w+', linha):
            yield match.group()


def filtrar_stop_words_stream(palavras_stream: Iterator[str]) -> Iterator[str]:
    """Filtra palavras sob demanda no fluxo contínuo."""
    for p in palavras_stream:
        if p not in STOP_WORDS_PT and len(p) > 1:
            yield p


def contar_frequencias_stream(palavras_stream: Iterator[str]) -> dict[str, int]:
    """Consome o fluxo contínuo sob demanda e agrega contagem final (sink)."""
    return dict(Counter(palavras_stream).most_common())


print("Executando pipeline em STREAMING (com geradores e yield):")
print("=" * 60)

# Pump / Fonte de dados que emite linha a linha sob demanda
fonte_linhas = (linha for linha in texto_bruto.strip().splitlines() if linha)

# Conexão dos Pipes: cada estágio é um gerador encadeado
stream_norm = normalizar_stream(fonte_linhas)
stream_tokens = tokenizar_stream(stream_norm)
stream_filtrado = filtrar_stop_words_stream(stream_tokens)

# Consumo pelo Sink final
resultado_stream = contar_frequencias_stream(stream_filtrado)

print("=" * 60)
print(f"📊 Top 5 palavras (Streaming sob demanda):")
for palavra, freq in list(resultado_stream.items())[:5]:
    print(f"   {palavra:.<25} {freq}")
print("\n💡 Vantagem: o pipeline processa item a item. Um log de 10 GB consumiria apenas poucos MB de RAM!")

Executando pipeline em STREAMING (com geradores e yield):
📊 Top 5 palavras (Streaming sob demanda):
   filtros.................. 3
   estilo................... 2
   processamento............ 2
   dados.................... 2
   arquitetura.............. 1

💡 Vantagem: o pipeline processa item a item. Um log de 10 GB consumiria apenas poucos MB de RAM!


## 4. A VIOLAÇÃO — Acoplamento entre filtros

Agora vamos violar a restrição fundamental do estilo: um filtro vai acessar
diretamente o estado interno de outro componente. Isso cria **acoplamento direto**,
comprometendo a substituição e a testabilidade independente.

In [7]:
# ❌ VERSÃO COM VIOLAÇÃO DE INDEPENDÊNCIA
# O filtro NormalizadorAcoplado guarda estado interno (idioma).
# O filtro ContadorAcoplado acessa diretamente esse estado por referência.

class NormalizadorAcoplado:
    """Filtro que guarda estado interno — viola a independência arquitetural."""
    def __init__(self, idioma='pt'):
        self.idioma = idioma  # estado que outros filtros vão acessar

    def __call__(self, texto: str) -> str:
        texto = texto.lower()
        texto = re.sub(r'[^\w\s]', '', texto)
        return texto


class ContadorAcoplado:
    """Filtro que depende diretamente do NormalizadorAcoplado — ACOPLAMENTO EFERENTE."""
    def __init__(self, normalizador: NormalizadorAcoplado):
        self.normalizador = normalizador  # ← dependência direta entre filtros!

    def __call__(self, palavras: list[str]) -> dict[str, int]:
        # Consulta o idioma diretamente no normalizador
        idioma = self.normalizador.idioma  # ← VIOLAÇÃO: acessa estado de outro filtro
        print(f"    ⚠️  ContadorAcoplado consultou idioma='{idioma}' do NormalizadorAcoplado")

        # Usa o idioma para decidir stop words
        stops = STOP_WORDS_PT if idioma == 'pt' else set()
        palavras_filtradas = [p for p in palavras if p not in stops and len(p) > 1]
        return dict(Counter(palavras_filtradas).most_common())


norm_acoplado = NormalizadorAcoplado(idioma='pt')
cont_acoplado = ContadorAcoplado(norm_acoplado)

print("Executando pipeline COM VIOLAÇÃO (acoplamento direto):")
print("=" * 60)
resultado_violado = pipeline(
    texto_bruto,
    norm_acoplado,
    tokenizar,
    cont_acoplado
)
print("=" * 60)
print(f"\n📊 Top 5 (versão acoplada):")
for palavra, freq in list(resultado_violado.items())[:5]:
    print(f"   {palavra:.<25} {freq}")

Executando pipeline COM VIOLAÇÃO (acoplamento direto):
  Filtro 1 (NormalizadorAcoplado): str → 
a arquitetura de software define a estrutura fundamental de um sistem...
  Filtro 2 (tokenizar): list → ['a', 'arquitetura', 'de', 'software', 'define', 'a', 'estrutura', 'fu...
    ⚠️  ContadorAcoplado consultou idioma='pt' do NormalizadorAcoplado
  Filtro 3 (ContadorAcoplado): dict → {'filtros': 3, 'estilo': 2, 'processamento': 2, 'dados': 2, 'arquitetu...

📊 Top 5 (versão acoplada):
   filtros.................. 3
   estilo................... 2
   processamento............ 2
   dados.................... 2
   arquitetura.............. 1


## 5. A consequência mensurável: Acoplamento Eferente

Medimos o impacto formal dessa violação utilizando a métrica de **Acoplamento Eferente ($C_e$)**:
quantas dependências externas cada filtro possui.

In [8]:
print("📏 Medindo acoplamento eferente (dependências externas de cada filtro)")
print("=" * 60)

# Versão CORRETA: cada filtro depende apenas da sua entrada legítima
filtros_corretos = {
    'normalizar': [],          # sem dependências externas (Ce = 0)
    'tokenizar': [],           # sem dependências externas (Ce = 0)
    'filtrar_stop_words': [],  # sem dependências externas (Ce = 0)
    'contar_frequencias': [],  # sem dependências externas (Ce = 0)
}
total_correto = 0
print("\n✅ Versão CORRETA (funções puras):")
for nome, deps in filtros_corretos.items():
    print(f"  {nome:.<25} {len(deps)} dependências {deps}")
    total_correto += len(deps)
print(f"  TOTAL Ce: {total_correto} dependências")

# Versão VIOLADA: ContadorAcoplado depende de NormalizadorAcoplado
filtros_violados = {
    'NormalizadorAcoplado': [],                      # sem dependências externas
    'tokenizar': [],                                 # sem dependências externas
    'ContadorAcoplado': ['NormalizadorAcoplado'],    # ← DEPENDE de outro filtro!
}
total_violado = 0
print("\n❌ Versão VIOLADA (acoplamento direto):")
for nome, deps in filtros_violados.items():
    print(f"  {nome:.<25} {len(deps)} dependência(s) {deps}")
    total_violado += len(deps)
print(f"  TOTAL Ce: {total_violado} dependência(s)")

print(f"\n📊 Impacto Arquitetural:")
print(f"   Acoplamento eferente total (correto):  {total_correto}")
print(f"   Acoplamento eferente total (violado):  {total_violado}")
print(f"   Aumento de {total_violado - total_correto} → a dependência impede a troca e teste isolado.")

📏 Medindo acoplamento eferente (dependências externas de cada filtro)

✅ Versão CORRETA (funções puras):
  normalizar............... 0 dependências []
  tokenizar................ 0 dependências []
  filtrar_stop_words....... 0 dependências []
  contar_frequencias....... 0 dependências []
  TOTAL Ce: 0 dependências

❌ Versão VIOLADA (acoplamento direto):
  NormalizadorAcoplado..... 0 dependência(s) []
  tokenizar................ 0 dependência(s) []
  ContadorAcoplado......... 1 dependência(s) ['NormalizadorAcoplado']
  TOTAL Ce: 1 dependência(s)

📊 Impacto Arquitetural:
   Acoplamento eferente total (correto):  0
   Acoplamento eferente total (violado):  1
   Aumento de 1 → a dependência impede a troca e teste isolado.


### 5.1 Impacto na modificabilidade: simulação de troca de filtros

Simulamos o cenário: **substituir o normalizador por uma versão em inglês**.
- Na versão correta, basta plugar o novo filtro — nenhum outro componente é afetado.
- Na versão violada, alterar o `NormalizadorAcoplado` exige obrigatoriamente revisar e alterar o `ContadorAcoplado`.

In [9]:
print("🔄 Simulação: trocar o normalizador por versão em inglês")
print("=" * 60)

def normalizar_en(texto: str) -> str:
    """Versão em inglês do normalizador."""
    return re.sub(r'[^\w\s]', '', texto.lower())

STOP_WORDS_EN = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'of', 'in', 'to', 'and', 'or', 'it', 'on', 'at'}

def filtrar_stop_words_en(palavras: list[str]) -> list[str]:
    return [p for p in palavras if p not in STOP_WORDS_EN and len(p) > 1]

print("\n✅ Versão CORRETA:")
print("   Componentes alterados: 2 (normalizar_en + filtrar_stop_words_en)")
print("   Componentes NÃO alterados: 2 (tokenizar + contar_frequencias)")
print("   Cada troca é INDEPENDENTE — nenhum filtro precisa saber do outro.")

print("\n❌ Versão VIOLADA:")
print("   Componentes que PRECISAM ser alterados: 2")
print("     1. NormalizadorAcoplado (trocar a lógica)")
print("     2. ContadorAcoplado (depende diretamente do normalizador!)")
print("   A dependência PROPAGOU: esse é o custo concreto do acoplamento.")

🔄 Simulação: trocar o normalizador por versão em inglês

✅ Versão CORRETA:
   Componentes alterados: 2 (normalizar_en + filtrar_stop_words_en)
   Componentes NÃO alterados: 2 (tokenizar + contar_frequencias)
   Cada troca é INDEPENDENTE — nenhum filtro precisa saber do outro.

❌ Versão VIOLADA:
   Componentes que PRECISAM ser alterados: 2
     1. NormalizadorAcoplado (trocar a lógica)
     2. ContadorAcoplado (depende diretamente do normalizador!)
   A dependência PROPAGOU: esse é o custo concreto do acoplamento.


### 5.2 Violação Realista em Sistemas de Produção: Mutabilidade de Payload

Em sistemas reais de software, violações arquiteturais raramente ocorrem por injeção explícita de classes em construtores (erros primários que ferramentas de análise estática e linters facilmente identificam).

Na prática de engenharia, a violação mais comum e danosa da independência no Pipe and Filter ocorre em **nível de dados**, através da **mutabilidade do payload**. Em linguagens como Python, dicionários e listas são passados por referência. Se um filtro altera diretamente (*in-place*) as chaves de um dicionário e o filtro seguinte assume implicitamente essa alteração, cria-se um **acoplamento temporal invisível**.

As assinaturas de função parecem perfeitamente desacopladas (`def filtro(dados): ...`), mas uma reordenação ou reutilização isolada do filtro causa falhas catastróficas em tempo de execução.

In [10]:
# ❌ VIOLAÇÃO SUTIL: Mutação in-place gerando acoplamento temporal invisível

def filtro_enriquecimento_mutavel(payload: dict) -> dict:
    """MUTA o dicionário original in-place adicionando metadados."""
    # Mutação direta no objeto compartilhado
    payload['idioma_detectado'] = 'pt'
    payload['texto_limpo'] = normalizar(payload['texto'])
    return payload


def filtro_contagem_dependente(payload: dict) -> dict:
    """Assume cegamente que 'texto_limpo' e 'idioma_detectado' já foram gerados."""
    # Falha se executado fora de ordem ou isolado!
    if 'texto_limpo' not in payload:
        raise KeyError("Acoplamento Temporal Quebrado! O filtro espera 'texto_limpo' gerado previamente.")
    
    palavras = tokenizar(payload['texto_limpo'])
    stops = STOP_WORDS_PT if payload.get('idioma_detectado') == 'pt' else set()
    payload['resultado'] = dict(Counter([p for p in palavras if p not in stops and len(p) > 1]).most_common(5))
    return payload


print("1. Execução na ordem esperada (funciona por coincidência temporal):")
payload_teste_1 = {'texto': "Arquitetura de software e estilo pipe and filter"}
filtro_enriquecimento_mutavel(payload_teste_1)
filtro_contagem_dependente(payload_teste_1)
print(f"   Resultado: {payload_teste_1['resultado']}")

print("\n2. Testando modificabilidade (tentando executar o filtro de contagem isoladamente):")
payload_teste_2 = {'texto': "Arquitetura de software e estilo pipe and filter"}
try:
    filtro_contagem_dependente(payload_teste_2)
except KeyError as err:
    print(f"   ❌ ERRO EM TEMPO DE EXECUÇÃO: {err}")
    print("   Explicação: As assinaturas pareciam independentes, mas havia acoplamento temporal oculto via mutação!")

# ✅ CORREÇÃO ARQUITETURAL: Imutabilidade de payload
def filtro_contagem_puro(texto: str) -> dict[str, int]:
    """Filtro puro: opera apenas sobre sua entrada legítima, sem mutar nem depender de chaves ocultas."""
    palavras = tokenizar(normalizar(texto))
    return dict(Counter(filtrar_stop_words(palavras)).most_common(5))

print("\n✅ Com imutabilidade e contratos explícitos, o acoplamento temporal é eliminado.")

1. Execução na ordem esperada (funciona por coincidência temporal):
   Resultado: {'arquitetura': 1, 'software': 1, 'estilo': 1, 'pipe': 1, 'and': 1}

2. Testando modificabilidade (tentando executar o filtro de contagem isoladamente):
   ❌ ERRO EM TEMPO DE EXECUÇÃO: "Acoplamento Temporal Quebrado! O filtro espera 'texto_limpo' gerado previamente."
   Explicação: As assinaturas pareciam independentes, mas havia acoplamento temporal oculto via mutação!

✅ Com imutabilidade e contratos explícitos, o acoplamento temporal é eliminado.


## 6. Pipe and Filter como classe reutilizável

Uma implementação orientada a objetos que formaliza o pipeline com interface fluente (`.adicionar()`) e capacidade de composição declarativa.

In [11]:
class Pipeline:
    """Pipeline reutilizável no estilo Pipe and Filter."""

    def __init__(self, nome: str = "Pipeline"):
        self.nome = nome
        self._filtros: list[Callable] = []

    def adicionar(self, filtro: Callable) -> 'Pipeline':
        """Adiciona um filtro ao pipeline. Retorna self para encadeamento fluente."""
        self._filtros.append(filtro)
        return self

    def executar(self, dado: Any) -> Any:
        """Executa todos os filtros em sequência."""
        resultado = dado
        for filtro in self._filtros:
            resultado = filtro(resultado)
        return resultado

    def __repr__(self) -> str:
        nomes = ' → '.join(f.__name__ if hasattr(f, '__name__') else str(f) for f in self._filtros)
        return f"{self.nome}: [{nomes}]"


# Montando pipelines reutilizáveis
pipe_completo = (
    Pipeline("Análise de frequência")
    .adicionar(normalizar)
    .adicionar(tokenizar)
    .adicionar(filtrar_stop_words)
    .adicionar(contar_frequencias)
)

pipe_simples = (
    Pipeline("Tokenização simples")
    .adicionar(normalizar)
    .adicionar(tokenizar)
)

print(f"Pipeline 1: {pipe_completo}")
print(f"Pipeline 2: {pipe_simples}")

print(f"\nResultado do Pipeline 1 (top 5 palavras):")
freq = pipe_completo.executar(texto_bruto)
for p, f in list(freq.items())[:5]:
    print(f"  {p}: {f}")

print(f"\nResultado do Pipeline 2 (primeiros 10 tokens):")
tokens = pipe_simples.executar(texto_bruto)
print(f"  {tokens[:10]}")

Pipeline 1: Análise de frequência: [normalizar → tokenizar → filtrar_stop_words → contar_frequencias]
Pipeline 2: Tokenização simples: [normalizar → tokenizar]

Resultado do Pipeline 1 (top 5 palavras):
  filtros: 3
  estilo: 2
  processamento: 2
  dados: 2
  arquitetura: 1

Resultado do Pipeline 2 (primeiros 10 tokens):
  ['a', 'arquitetura', 'de', 'software', 'define', 'a', 'estrutura', 'fundamental', 'de', 'um']


## 7. Medição Empírica dos Trade-offs: Pipeline vs. Abordagem Monolítica com `timeit`

Um princípio fundamental do design de arquitetura é mensurar empiricamente os custos de uma decisão com o mesmo rigor aplicado aos seus benefícios.

A reusabilidade, testabilidade e modificabilidade do **Pipe and Filter** cobram um pedágio computacional: encadear múltiplos estágios, realizar chamadas de função adicionais e passar estruturas de dados entre conectores impõe uma sobrecarga computacional de CPU e tempo de execução.

Para mensurar esse trade-off com precisão e sem interferência do coletor de lixo (*garbage collector*) ou variações do sistema operacional, utilizamos o módulo padrão **`timeit`**.

Comparamos o **Pipeline modular** contra uma implementação puramente **monolítica** (função fechada em loop único).

In [12]:
# Simulação com volume maior de dados (representando análise de logs)
massa_logs = texto_bruto * 20

# Implementação puramente monolítica (tudo em um único loop enxuto, sem pipes nem listas intermediárias)
def processamento_monolitico(texto: str) -> dict[str, int]:
    """Processamento monolítico: todas as etapas ocorrem em um único escopo sem conexões modulares."""
    texto_limpo = re.sub(r'[^\w\s]', '', texto.lower())
    palavras = [p for p in texto_limpo.split() if p not in STOP_WORDS_PT and len(p) > 1]
    return dict(Counter(palavras).most_common())


# Microbenchmark rigoroso com timeit
NUMERO_ITERACOES = 500

tempo_pipeline = timeit.timeit(
    stmt="pipe_completo.executar(massa_logs)",
    globals=globals(),
    number=NUMERO_ITERACOES
)

tempo_monolitico = timeit.timeit(
    stmt="processamento_monolitico(massa_logs)",
    globals=globals(),
    number=NUMERO_ITERACOES
)

diferenca_pct = ((tempo_pipeline - tempo_monolitico) / tempo_monolitico) * 100

print(f"⏱️ Benchmark empírico ({NUMERO_ITERACOES} iterações):")
print("=" * 60)
print(f"  Abordagem Monolítica:           {tempo_monolitico:.4f} segundos")
print(f"  Arquitetura Pipe and Filter:   {tempo_pipeline:.4f} segundos")
print("=" * 60)
print(f"📊 Análise do Trade-off de Desempenho:")
print("   A abordagem monolítica concentra as transformações em um único escopo de função,")
print("   enquanto a arquitetura Pipe and Filter distribui o processamento em múltiplos filtros")
print("   e chamadas encadeadas. Essa sobrecarga de conectores é o imposto pago conscientemente")
print("   para adquirir: modularidade, reusabilidade e capacidade de teste unitário isolado.")

⏱️ Benchmark empírico (500 iterações):
  Abordagem Monolítica:           0.4275 segundos
  Arquitetura Pipe and Filter:   0.4287 segundos
📊 Análise do Trade-off de Desempenho:
   A abordagem monolítica concentra as transformações em um único escopo de função,
   enquanto a arquitetura Pipe and Filter distribui o processamento em múltiplos filtros
   e chamadas encadeadas. Essa sobrecarga de conectores é o imposto pago conscientemente
   para adquirir: modularidade, reusabilidade e capacidade de teste unitário isolado.


## 8. Conclusão e Esclarecimento sobre Serialização

O estilo arquitetural **Pipe and Filter** oferece benefícios claros:

- **Reusabilidade:** os mesmos filtros podem ser combinados em pipelines distintos sem alteração de código.
- **Modificabilidade:** a troca ou atualização de um filtro não afeta os demais, desde que a interface do pipe seja mantida.
- **Testabilidade:** cada filtro pode ser verificado isoladamente através de testes unitários determinísticos com entradas e saídas controladas.
- **Processamento sob demanda:** com geradores (`yield`), suporta fluxos contínuos de dados com consumo de memória constante.

Por outro lado, o padrão impõe custos e trade-offs concretos:

- **Sobrecarga de chamadas e conectores:** conforme comprovado pela medição com `timeit`, a modularidade exige mais ciclos de processamento que um loop monolítico fechado.
- **Baixa adequação para sistemas interativos:** o fluxo unidirecional não suporta facilmente ciclos de feedback em tempo real com o usuário final.

### 📌 Esclarecimento sobre o custo de "Serialização"

> **Nota técnica fundamental:** No nosso exemplo em código, **não sofremos com o custo de serialização** pois os dados trafegam na memória do mesmo processo, via chamadas de função e ponteiros de objetos.
> 
> No entanto, em uma **arquitetura de microserviços distribuídos** ou em pipelines tradicionais do Unix entre processos distintos do sistema operacional, esse custo seria real e substancial: cada pipe exigiria serializar os dados (para JSON, Protocol Buffers ou streams de bytes) na saída de um filtro e desserializá-los na entrada do próximo, gerando latência de rede e alto consumo de CPU.
> 
> A regra de ouro: conheça a escala e o contexto do seu sistema para decidir se o ganho em desacoplamento compensa os custos operacionais da arquitetura.